### Core Strategy 3: Refiner Versus Crack Spread

In [ ]:
import time
import pandas as pd

import omega
from omega import start_loop, Stock, Future, MarketOrder
import logging

In [ ]:
window = 21
thresh = 2
duration = 252

### Create an Omega trading app
This code creates an instance of an Omega trading app, connecting to the trading server at `127.0.0.1` and port `7497`. It also sets the client ID to 10 and specifies the account number as "DU7129120" for the trading session.

In [ ]:
# You must run `start_loop` when using Omega from Jupyter Notebook
start_loop()

# To debug, instantiate Omega(log_level=logging.DEBUG)
app = omega.Omega()

### Data Acquisition
First we set up the machinery for acquiring historical data. The function `fetch_historical_data` retrieves historical market data for a list of contracts over a specified duration. It uses the `app.get_historical_data_for_many` method to fetch the data, specifying the contracts, duration (in days), bar size (defaulting to "1 day"), and the type of data to show ("MIDPOINT"). The function takes three parameters: `contracts`, a list of contract objects; duration, the number of days for which to fetch data; and an optional bar_size parameter that defaults to "1 day". This function returns the historical data for the provided contracts.

In [ ]:
def fetch_historical_data(contracts, duration, bar_size="1 day"):
    return app.get_historical_data_for_many(
        contracts=contracts,
        end_date_time="",
        duration=f"{duration} D",
        bar_size=bar_size,
        what_to_show="MIDPOINT",
    )

The code snippet creates four contract objects representing different financial instruments:

1. `psx = Stock("PSX", "SMART", "USD")`: This creates a stock contract for Phillips 66 (symbol "PSX"), traded on the SMART exchange, with the currency set to USD.
2. `ho = Future("HO", "202409", "NYMEX")`: This creates a futures contract for heating oil (symbol "HO"), with an expiration date in September 2024, traded on the NYMEX exchange.
3. `rb = Future("RB", "202409", "NYMEX")`: This creates a futures contract for reformulated gasoline blendstock (symbol "RB"), with an expiration date in September 2024, traded on the NYMEX exchange.
4. `cl = Future("CL", "202409", "NYMEX")`: This creates a futures contract for crude oil (symbol "CL"), with an expiration date in September 2024, traded on the NYMEX exchange.

These contract objects can be used to fetch market data or place trades.

In [ ]:
psx = Stock("PSX", "SMART", "USD")
ho = Future("HO", "202501", "NYMEX")
rb = Future("RB", "202501", "NYMEX")
cl = Future("CL", "202501", "NYMEX")

We fetch historical market data for the specified contracts (`psx`, `ho`, `rb`, `cl`) over a given duration using the `fetch_historical_data` function. The `historical_data` variable stores the returned data.

In [ ]:
historical_data = fetch_historical_data(
    contracts=[psx, ho, rb, cl],
    duration=duration
)

In [ ]:
historical_data

Omega returns the symbol of the requested contract in each row of data. This makes it easy to pivot the resulting DataFrame to put the closing prices of each contract in each column.

In [ ]:
data = historical_data.pivot(columns="symbol", values="close").dropna()

In [ ]:
data

### Building the spread
The 3:2:1 crack spread calculation starts with the spot price for two barrels of gasoline, added to the spot price for one barrel of heating oil, and then subtracts the spot price for three barrels of WTI crude oil. We use the spot month RBOB gasoline per-gallon price multiplied by 42 to reach a barrel, and the spot month NY heating oil per-gallon price multiplied by 42 to reach a barrel. WTI crude is already quoted in dollars per barrel. The resulting value is then divided by three.

In [ ]:
# Compute price per barrel
data.RB *= 42  # equiv to data.RB = data.RB * 42
data.HO *= 42

# Construct the crack spread
data["crack_spread"] = 2 * data.RB + data.HO - 3 * data.CL
data["crack_spread"] /= 3

Plot the spread over time.

In [ ]:
data.crack_spread.plot()

We add a new column, `crack_spread_rank` to the `data` DataFrame by calculating the rolling rank of the `crack_spread` values. Using a specified window size (`window`), we compute the rank of each value as a percentage of the rolling window. This helps us understand the relative position of the crack spread within the specified rolling window.

In [ ]:
data["crack_spread_rank"] = data.crack_spread.rolling(window).rank(pct=True)

Plot the spread over time.

In [ ]:
data.crack_spread_rank.plot()

Compute the percentile rank for the refiner's price

In [ ]:
data["refiner_rank"] = data.PSX.rolling(window).rank(pct=True)

Plot the spread over time.

In [ ]:
data.refiner_rank.plot()

We create a new column, `rank_spread` in the `data` DataFrame by subtracting the `crack_spread_rank` from the `refiner_rank`. This calculation gives us the difference between the refiner rank and the crack spread rank, allowing us to analyze the relative performance or relationship between these two metrics.

In [ ]:
data["rank_spread"] = data.refiner_rank - data.crack_spread_rank

Plot the spread over time.

In [ ]:
data.rank_spread.plot()

We calculate a rolling z-score for the `rank_spread` column in the data DataFrame. First, we create a rolling window object roll with the specified window size. Then, we compute the z-score by subtracting the rolling mean from the `rank_spread` values and dividing by the rolling standard deviation. Finally, we assign the most recent z-score value to the variable signal. This is our trading signal.

In [ ]:
roll = data.rank_spread.rolling(window)
zscore = ((data.rank_spread - roll.mean()) / roll.std())
signal = zscore.iloc[-1]

Plot the z-score over time.

In [ ]:
zscore.plot()

In [ ]:
signal

### Parameterize the strategy
We want to trade this strategy on an intraday basis. To do so, we'll take the code above, tweak it slightly, and put it all in a single function. That way we can run the entire thing in a loop to check if the signal meets our criteria for a trade.

We define a function `generate_signal` to compute a trading signal for a refiner stock and futures contracts. First, we create contract objects for the refiner stock and three futures (heating oil, gasoline, and crude oil) based on the provided expiration date. We then fetch historical data for these contracts, using a duration of 2 days and a bar size of 30 seconds. We pivot and clean the data, converting prices for gasoline and heating oil to a per-barrel basis and constructing the crack spread. Next, we calculate rolling percentile ranks for the crack spread and refiner stock, compute the rank spread, and then derive the z-score of the rank spread. Finally, we return the most recent z-score as the trading signal.

In [ ]:
def generate_signal(refiner, expiration, window=120):
    """
    Generate a trading signal based on the crack spread and refiner stock ranks.
    
    Parameters
    ----------
    refiner : str
        The symbol of the refiner stock.
    expiration : str
        The expiration date of the futures contracts.
    window : int, optional
        The rolling window size for ranking and z-score calculation, by default 120.
    
    Returns
    -------
    float
        The most recent z-score of the rank spread.
    """
    ref = Stock(refiner, "SMART", "USD")
    ho = Future("HO", expiration, "NYMEX")
    rb = Future("RB", expiration, "NYMEX")
    cl = Future("CL", expiration, "NYMEX")
    
    historical_data = fetch_historical_data(
        contracts=[ref, ho, rb, cl],
        duration=2,
        bar_size="30 secs"
    )
    
    data = (
        historical_data
        .pivot(
            columns="symbol", 
            values="close"
        )
        .dropna()
    )
    
    data.RB *= 42
    data.HO *= 42

    # Construct the crack spread
    data["crack_spread"] = 2 * data.RB + data.HO - 3 * data.CL
    data["crack_spread"] /= 3
    
    data["crack_spread_rank"] = (
        data
        .crack_spread
        .rolling(window)
        .rank(pct=True)
    )
    
    data["refiner_rank"] = (
        data[refiner]
        .rolling(window)
        .rank(pct=True)
    )
    
    data["rank_spread"] = data.refiner_rank - data.crack_spread_rank
    
    roll = data.rank_spread.rolling(window)
    
    z_score = ((data.rank_spread - roll.mean()) / roll.std())
    
    return z_score.iloc[-1]

We continuously monitor the trading signal for the refiner by calling the `generate_signal` function within a while loop. We check if we currently hold a position in the refiner and print the signal and holding status. We then sleep for 30 seconds to match the frequency of the historical data retrieval. If the signal indicates a strong buy (below our threshold) and we don't already hold the position, we print a message and send a buy order. Conversely, if the signal turns positive and we are holding the position, we print a message and exit the position. Similarly, if the signal indicates a strong sell (above our threshold) and we don't hold the position, we print a message and send a sell order. If the signal turns negative and we are holding a short position, we print a message and exit the position.

In [ ]:
refiner = "PSX"
expiration = "202501"
contracts_to_trade = 50

This function, get_position_by_symbol, retrieves the position of a specified stock symbol from a list of positions provided by app.positions(). It iterates through each position, and if the symbol attribute of a position's contract matches the input symbol, it returns the corresponding position value. If no match is found, the function returns 0.0, indicating no position for the given symbol.

In [ ]:
def get_position_by_symbol(symbol):
    for pos in app.positions():
        if pos.contract.symbol == symbol:
            return pos.position
    return 0.0

In [ ]:
while True:

    signal = generate_signal(refiner, expiration)

    # If the position exists in the account, get the
    # share quantity, else return 0.0
    holding = get_position_by_symbol(refiner)
    
    print(f"Signal is {signal}. Holding {refiner}: {holding}")
    
    # Sleep for 30 seconds since we're pulling 
    # 30 second historic data
    time.sleep(30)

    # Enter a long position if the signal is below our
    # threshold and we do not currently hold the position
    if signal <= -thresh and holding == 0.0:
        print(f"Enter long: Signal is {signal}. Holding {refiner}: {holding}")
        order = MarketOrder(action="BUY", totalQuantity=contracts_to_trade)
        app.order(contract, order)

    # Exit the position if the signal exceeds 0 and we
    # are holding a long position
    elif signal >= 0 and holding > 0:
        print(f"Exit long: Signal is {signal}. Holding {refiner}: {holding}")
        app.order_target_percent(
            contract=contract, 
            order_type=MarketOrder,
            target=0
        )

    # Enter a short position if the signal exceeds our
    # threshold and we do not currently hold the position
    if signal >= thresh and holding == 0.0:
        print(f"Enter short: Signal is {signal}. Holding {refiner}: {holding}")
        order = MarketOrder(action="SELL", totalQuantity=contracts_to_trade)
        app.order(contract, order)

    # Exit the position if the signal falls below 0 and we
    # are holding the position
    elif signal <= 0 and holding < 0:
        print(f"Exit short: Signal is {signal}. Holding {refiner}: {holding}")
        app.order_target_percent(
            contract=contract, 
            order_type=MarketOrder,
            target=0
        )

Since we are looking at 30 second bars, we want enough data to compute a meaningful set of signals. That's why we use a 120 period window (30 seconds x 120 bars = 1 hour